# AI2002 - Group 5 - Review Chatbot for Hotel Question Answering

## Đề tài: Xây dựng Chatbot thông minh hỗ trợ trả lời câu hỏi và tư vấn khách sạn dựa trên tập dữ liệu đánh giá TripAdvisor (Hotel Review Question Answering)

## 0. Cài đặt môi trường

Với dự án nghiên cứu về **Review Chatbot for Hotel Question Answering**, nhóm nghiên cứu đã thiết lập các thư viện và công cụ sau:

1. **Pandas**: Dùng để cấu trúc hóa dữ liệu lớn từ file .csv thành DataFrame, thực hiện tiền xử lý, lọc theo địa phương/ngôn ngữ và biến đổi dữ liệu đánh giá khách sạn.
2. **NumPy**: Hỗ trợ các phép toán ma trận, vector hóa các trọng số điểm số và tính toán số học hiệu năng cao.
3. **SQLite3**: Quản trị cơ sở dữ liệu quan hệ cục bộ (`chatbot.db`), giúp lưu trữ có cấu trúc các bảng thực thể khách sạn, các khía cạnh dịch vụ và truy vấn nhanh chóng bằng câu lệnh SQL.
4. **Matplotlib / Seaborn**: Hỗ trợ trực quan hóa phân bố điểm đánh giá, tỉ lệ đánh giá theo tỉnh thành/hạng sao, và thống kê độ dài review.
5. **NLTK & Scikit-learn**: Cung cấp công cụ xử lý ngôn ngữ tự nhiên (NLP) cơ bản, tách từ, loại bỏ từ dừng (stop words), xây dựng ma trận đặc trưng văn bản (TF-IDF Vectorizer) và đo lường độ tương đồng ngữ nghĩa (Cosine Similarity) phục vụ cơ chế truy xuất câu trả lời (Question Answering).

In [ ]:
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import nltk

# Kiểm tra phiên bản tất cả thư viện
print(f"{'Thư viện':<20} {'Phiên bản':>15}")
print("-" * 37)
print(f"{'Python':<20} {os.sys.version.split()[0]:>15}")
print(f"{'Pandas':<20} {pd.__version__:>15}")
print(f"{'NumPy':<20} {np.__version__:>15}")
print(f"{'Matplotlib':<20} {matplotlib.__version__:>15}")
print(f"{'Seaborn':<20} {sns.__version__:>15}")
print(f"{'Scikit-learn':<20} {sklearn.__version__:>15}")
print(f"{'NLTK':<20} {nltk.__version__:>15}")
print(f"{'SQLite3':<20} {sqlite3.sqlite_version:>15}")
print(f"{'Regex (re)':<20} {re.__version__ if hasattr(re, '__version__') else 'built-in':>15}")

# Khởi tạo kết nối SQLite
conn = sqlite3.connect('Data (Dataset, Data Frame, Chart Images)/chatbot.db')

print("\n--- Môi trường khởi tạo thành công ---")

Thư viện                   Phiên bản
-------------------------------------
Python                       3.12.14
Pandas                         3.0.5
NumPy                          2.5.3
Matplotlib                    3.11.1
Seaborn                       0.13.2
Scikit-learn                   1.9.0
NLTK                          3.10.3
SQLite3                       3.53.4
Regex (re)                     2.2.1

--- Môi trường cho Hotel Review Chatbot đã được khởi tạo thành công ---


## 1. Nhập và Kiểm tra Dataset (Dataset Loading & Exploration)

Trong phần này, nhóm sẽ thực hiện các bước chuẩn bị và khảo sát ban đầu:
1. **Tải tập dữ liệu nghiên cứu**: Đọc tập dữ liệu đánh giá khách sạn từ file CSV (`tripadvisor_review_hotel_dataset.csv`). Chương trình tự động kiểm tra sự tồn tại của file ở thư mục hiện tại, thư mục lưu trữ dữ liệu `Data (Dataset, Data Frame, Chart Images)/` hoặc trên Google Drive.
2. **Chuyển đổi vào Pandas DataFrame**: Sử dụng bộ mã hóa UTF-8 để đảm bảo hiển thị chuẩn xác tiếng Việt và các ngôn ngữ quốc tế.
3. **Kiểm tra cấu trúc và các trường thông tin quan trọng**:
   - **Thông tin cơ sở lưu trú**: `hotel_name`, `hotel_province`, `hotel_address`, `hotel_star`.
   - **Điểm đánh giá và các khía cạnh dịch vụ**: `normalized_score`, `Value`, `Rooms`, `Location`, `Cleanliness`, `Service`, `Sleep_Quality`.
   - **Dữ liệu văn bản phục vụ Chatbot Hỏi Đáp (QA)**: `normalized_title` (tiêu đề đánh giá), `normalized_content` (nội dung đánh giá chi tiết), `Word_count`, `language_code`, `language`.
   - **Bối cảnh chuyến đi & Thời gian**: `trip_type`, `Date`, `month`, `year`.

In [ ]:
import os
import pandas as pd

# Tạo thư mục lưu trữ biểu đồ trực quan hóa nếu chưa tồn tại
os.makedirs('charts_img', exist_ok=True)

# Danh sách các đường dẫn ứng viên của dataset (hỗ trợ cả Local và Google Colab)
candidate_paths = [
    'Data (Dataset, Data Frame, Chart Images)/tripadvisor_review_hotel_dataset.csv',
    'tripadvisor_review_hotel_dataset.csv',
    '/content/drive/MyDrive/Data (Dataset, Data Frame, Chart Images)/tripadvisor_review_hotel_dataset.csv',
    '/content/drive/MyDrive/tripadvisor_review_hotel_dataset.csv',
    '/content/tripadvisor_review_hotel_dataset.csv'
]

file_path = None
for path in candidate_paths:
    if os.path.exists(path):
        file_path = path
        print(f"Đã tìm thấy dataset tại: {file_path}")
        break

if file_path is None:
    file_path = candidate_paths[0]
    print(f"Cảnh báo: Chưa tìm thấy file dataset. Hãy đảm bảo file '{os.path.basename(file_path)}' đã được tải lên.")

# Tiến hành đọc file CSV
try:
    df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)
    print(f"Kích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
    print("\nCác cột dữ liệu phục vụ nghiên cứu:")
    print(list(df.columns))
    
    # Hiển thị 5 dòng đầu tiên
    print("\nXem trước 5 dòng đầu tiên:")
    display(df.head())
except Exception as e:
    print(f"Lỗi khi đọc file CSV: {e}")